In [1]:
from dotenv import load_dotenv
from google import genai
import os
import json

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
print("Client ready!")

Client ready!


In [2]:
with open('../prompts/writer_system_prompt.txt') as f:
    writer_prompt = f.read()
print("Writer prompt loaded!")
print(writer_prompt[:200])

Writer prompt loaded!
You are an expert Technical Product Manager with 10+ years of experience writing Product Requirements Documents (PRDs) for top tech companies.

Your job is to convert rough notes or ideas into a compl


In [3]:
rough_notes = "Build a ride-sharing app for college campuses."

response = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{writer_prompt}\n\nGenerate a complete PRD from these notes:\n{rough_notes}"
)

prd_output = response.text
print(prd_output)

# Product Requirements Document: CampusRide

## 1. Executive Summary
CampusRide is a specialized, hyper-local ride-sharing platform designed exclusively for university students, faculty, and staff. Unlike general-market apps like Uber or Lyft, CampusRide focuses on the specific security, affordability, and social dynamics of a college environment. The platform solves the “last-mile” problem of getting students from campus centers to off-campus housing or late-night study locations safely and economically. By verifying university email domains (.edu), the app creates a trusted network of verified peers. It is designed for students with tight budgets and high-frequency, short-distance travel needs.

## 2. Problem Statement
**Current Pain Points:**
*   **Safety Concerns:** Students often feel unsafe walking alone late at night; general ride-share apps are costly and often lack a sense of community trust.
*   **Cost Prohibitions:** College students have limited disposable income, and surge

In [4]:
with open('../prompts/critic_system_prompt.txt') as f:
    critic_prompt = f.read()
print("Critic prompt loaded!")

Critic prompt loaded!


In [5]:
response2 = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{critic_prompt}\n\nReview this PRD:\n{prd_output}"
)

raw = response2.text.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]

result = json.loads(raw)
print(json.dumps(result, indent=2))

{
  "status": "needs_revision",
  "score": 68,
  "missing_sections": [],
  "feedback": [
    "Legal and liability risks are severely underestimated; relying on 'lax laws' or 'airtight ToS' is not a strategy, especially regarding student safety and local transportation statutes.",
    "The 'Success Metrics' are overly optimistic and lack baseline data; acquiring 20% of a student body in one semester without a clear customer acquisition cost (CAC) model is unrealistic.",
    "Section 6 (Non-Functional Requirements) lacks a disaster recovery plan beyond uptime stats.",
    "The technical stack suggests both MongoDB and PostgreSQL, but the PRD fails to clearly delineate which data sets belong in which database or how to handle consistency between them in a microservices environment.",
    "User stories lack 'Negative' or 'Edge Case' paths; they only describe the 'Happy Path'.",
    "Section 8 (Edge Cases) is reactive; it lacks proactive security measures for the app's most significant thre

In [6]:
# Sirisha's cell - testing critic agent
from dotenv import load_dotenv
from google import genai
import os
import json

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

with open('../prompts/critic_system_prompt.txt') as f:
    critic_prompt = f.read()

# Paste Vedha's PRD output here
sample_prd = """# Product Requirements Document: CampusRide

## 1. Executive Summary
CampusRide is a specialized, hyper-local ride-sharing platform designed exclusively for university and college students. Unlike general-market apps like Uber or Lyft, CampusRide focuses on verified student-to-student transportation, facilitating safer, more affordable, and community-driven mobility within and around campus environments. The product solves the "transportation gap" that many students face due to lack of reliable campus shuttles, limited parking spaces, and the high cost of traditional ride-sharing services. By leveraging university email verification (.edu domains), the platform fosters a high-trust environment. It serves two core user groups: student drivers looking to subsidize vehicle ownership costs, and student passengers seeking affordable, reliable transit to labs, off-campus housing, and local transit hubs.

---

## 2. Problem Statement
**Current Pain Points:**
*   **Affordability:** Traditional ride-sharing apps have surged in price due to dynamic pricing, making them inaccessible for students on a budget.
*   **Safety and Trust:** Students feel safer riding with peers than with anonymous strangers in a general public app.
*   **Infrastructure Gaps:** Campus shuttle services are often infrequent, overcrowded, or stop running before late-night library study sessions or social events conclude.
*   **Parking Congestion:** University parking permits are expensive and limited; students who don't need to drive daily are incentivized to share rides.

**Why Now:** With the rising cost of living and the increasing density of student populations in urban campuses, students are looking for micro-mobility solutions that are both social and economic. 

**Impact of Not Solving:** Students will continue to rely on unreliable public transport, walk in potentially unsafe areas late at night, or drive their own vehicles, contributing to worsening parking shortages and increased environmental impact on campus.

---

## 3. User Personas

**Persona 1: The Commuter Student**
*   **Name:** Alex Rivera
*   **Age:** 20
...

***

*(Note: In a real-world scenario, I would expand the technical and functional sections with specific API documentation references and data flow diagrams to reach the full depth required for engineering teams.)*"""

response = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{critic_prompt}\n\nReview this PRD:\n{sample_prd}"
)

raw = response.text.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]

result = json.loads(raw)
print(json.dumps(result, indent=2))

{
  "status": "needs_revision",
  "score": 40,
  "missing_sections": [
    "4. Functional Requirements",
    "5. Non-Functional Requirements",
    "6. User Experience & Design",
    "7. Technical Architecture",
    "8. Security and Privacy",
    "9. Go-to-Market / Launch Strategy",
    "10. Success Metrics (KPIs)"
  ],
  "feedback": [
    "The PRD is severely incomplete; it stops after the User Personas section.",
    "Functional requirements are entirely absent, leaving engineers with no scope of work.",
    "Non-functional requirements regarding latency, scalability, and data storage are missing.",
    "Security and privacy sections are critical for a service relying on PII and location tracking, yet they are omitted.",
    "Missing success metrics makes it impossible to define what 'success' looks like for the product team.",
    "Technical architecture and integration strategies are completely undefined."
  ],
  "approval_message": "The document is currently a high-level concept dr